# 🎙️ Colab TTS Worker — OmniVoice
Kết nối server qua WebSocket, xử lý TTS task bằng [k2-fsa/OmniVoice](https://github.com/k2-fsa/OmniVoice).

In [ ]:
#@title 1. Cài đặt & Load Model
import os, sys, time

# Kiểm tra GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'✅ GPU: {gpu_name}')
else:
    print('⚠️ KHÔNG có GPU! Worker sẽ chạy rất chậm trên CPU.')

# Cài đặt OmniVoice (Colab đã có sẵn torch, torchaudio)
# QUAN TRỌNG: --no-deps để tránh downgrade PyTorch sang CPU-only
print('📦 Đang cài đặt OmniVoice...')
!pip install -q --upgrade "transformers>=5.3"
!pip install -q omnivoice --no-deps
!pip install -q soundfile pydub soxr websockets httpx

# Load model
print('🔄 Đang tải model OmniVoice...')
t0 = time.time()
from omnivoice import OmniVoice
import soundfile as sf

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice',
    device_map=DEVICE,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
load_ms = round((time.time() - t0) * 1000, 1)
print(f'✅ Model loaded trên {DEVICE} trong {load_ms:.0f}ms')

In [ ]:
#@title 2. WebSocket Worker — Kết nối Server & Xử lý Task
import asyncio
import json
import io
import hashlib
import tempfile
import httpx
import websockets
import torch
import soundfile as sf

# ── CONFIG ────────────────────────────────────────────────
SERVER_URL = ''  #@param {type:"string"}
EMAIL = ''       #@param {type:"string"}

SAMPLE_RATE = 24000  # OmniVoice output 24kHz
REF_CACHE_DIR = '/tmp/omnivoice_refs'
os.makedirs(REF_CACHE_DIR, exist_ok=True)

WS_URL = SERVER_URL.replace('http', 'ws') + '/ws/worker'


def run_tts(text: str, ref_audio_path: str, ref_text: str = '') -> bytes:
    """Chạy OmniVoice TTS voice cloning.

    Args:
        text: Văn bản cần tổng hợp giọng nói.
        ref_audio_path: Đường dẫn file audio tham chiếu (3-10 giây).
        ref_text: Transcript của audio tham chiếu (tùy chọn).

    Returns:
        WAV audio bytes (24kHz) của giọng nói được tổng hợp.
    """
    kwargs = dict(text=text, ref_audio=ref_audio_path)
    if ref_text:
        kwargs['ref_text'] = ref_text

    audios = model.generate(**kwargs)

    # audios là list[np.ndarray] tại 24000 Hz
    buf = io.BytesIO()
    sf.write(buf, audios[0], SAMPLE_RATE, format='WAV')
    return buf.getvalue()


async def download_ref_audio(http_client: httpx.AsyncClient, voice_url: str) -> str:
    """Tải audio tham chiếu, cache theo URL hash để tránh tải lại."""
    full_url = voice_url if voice_url.startswith('http') else f'{SERVER_URL}{voice_url}'
    url_hash = hashlib.md5(full_url.encode()).hexdigest()[:12]
    cached = os.path.join(REF_CACHE_DIR, f'ref_{url_hash}.wav')

    if os.path.exists(cached):
        print(f'  📁 Ref audio cache hit: {url_hash}')
        return cached

    print(f'  ⬇️ Downloading ref audio: {full_url}')
    resp = await http_client.get(full_url, timeout=30)
    resp.raise_for_status()
    with open(cached, 'wb') as f:
        f.write(resp.content)
    return cached


async def worker_loop():
    """Main worker loop: kết nối WebSocket, lắng nghe task, tự reconnect."""
    retry_delay = 5
    max_delay = 60

    async with httpx.AsyncClient() as http_client:
        while True:
            try:
                print(f'🔌 Đang kết nối: {WS_URL} ...')
                async with websockets.connect(
                    WS_URL,
                    open_timeout=30,
                    close_timeout=10,
                    ping_interval=30,
                    ping_timeout=20
                ) as ws:
                    # Reset retry delay khi kết nối thành công
                    retry_delay = 5

                    # Đăng ký với server
                    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
                    await ws.send(json.dumps({
                        'action': 'register',
                        'email': EMAIL,
                        'gpu': gpu_name
                    }))
                    print(f'✅ Đã đăng ký worker: {EMAIL} (GPU: {gpu_name})')

                    # Thông báo IDLE
                    await ws.send(json.dumps({'action': 'status', 'status': 'IDLE'}))

                    while True:
                        raw = await ws.recv()
                        data = json.loads(raw)
                        action = data.get('action')

                        if action == 'run_tts':
                            task_id = data['task_id']
                            text = data['text']
                            voice_url = data['voice_api_url']

                            print(f'📥 Task nhận: {task_id}')
                            print(f'   Text: "{text[:80]}..."' if len(text) > 80 else f'   Text: "{text}"')

                            try:
                                # Đánh dấu BUSY
                                await ws.send(json.dumps({'action': 'status', 'status': 'BUSY'}))

                                # Tải audio tham chiếu
                                ref_path = await download_ref_audio(http_client, voice_url)

                                # Chạy TTS
                                t_start = time.time()
                                result_audio = run_tts(text, ref_path)
                                tts_ms = round((time.time() - t_start) * 1000, 1)

                                # Upload kết quả
                                upload_url = f'{SERVER_URL}/api/tasks/{task_id}/complete'
                                files = {'audio': ('result.wav', result_audio, 'audio/wav')}
                                upload_resp = await http_client.post(upload_url, files=files, timeout=120)
                                upload_resp.raise_for_status()

                                # Thông báo hoàn thành
                                await ws.send(json.dumps({
                                    'action': 'task_completed',
                                    'task_id': task_id
                                }))

                                audio_duration = len(result_audio) / (SAMPLE_RATE * 2)  # ước tính
                                print(f'✅ Task hoàn thành: {task_id} (TTS: {tts_ms:.0f}ms, ~{audio_duration:.1f}s audio)')

                            except Exception as e:
                                print(f'❌ Task thất bại: {e}')
                                try:
                                    await ws.send(json.dumps({
                                        'action': 'task_failed',
                                        'task_id': task_id,
                                        'error': str(e)
                                    }))
                                except Exception:
                                    pass
                            finally:
                                try:
                                    await ws.send(json.dumps({'action': 'status', 'status': 'IDLE'}))
                                except Exception:
                                    pass

                        elif action == 'ping':
                            await ws.send(json.dumps({'action': 'pong'}))

                        elif action == 'pong':
                            pass  # Heartbeat response

                        elif action == 'shutdown':
                            print('🛑 Server yêu cầu shutdown.')
                            return

            except (websockets.ConnectionClosed, OSError) as e:
                print(f'⚠️ Mất kết nối: {e}. Thử lại sau {retry_delay}s...')
                await asyncio.sleep(retry_delay)
                retry_delay = min(retry_delay * 2, max_delay)
            except Exception as e:
                print(f'❌ Lỗi không mong đợi: {e}. Thử lại sau {retry_delay}s...')
                await asyncio.sleep(retry_delay)
                retry_delay = min(retry_delay * 2, max_delay)

print('✅ Worker loop sẵn sàng. Đang khởi chạy...')
await worker_loop()